In [ ]:
# Authenticate with Hugging Face Hub to access gated models
!hf auth login

In [ ]:
# Install dependencies for model quantization and acceleration
# - accelerate: enables efficient model loading and inference
# - bitsandbytes: provides 4-bit quantization for memory-efficient model inference
!pip install -q accelerate
!pip install -U -q bitsandbytes

#for tts

In [ ]:
# Install Parler TTS for text-to-speech synthesis
!pip install -q git+https://github.com/huggingface/parler-tts.git

#for translation

In [ ]:
# Clone IndicTrans2 repository for English-to-Indian language translation
!git clone https://github.com/AI4Bharat/IndicTrans2.git
%cd IndicTrans2/huggingface_interface

# NOTE: %cd makes the directory change permanent in Jupyter notebooks
# Install all dependencies required for translation models
!source install.sh

#restart session after this !!!

In [ ]:
# !pip install -q -U transformers==4.55.4

In [ ]:
# Setup device (GPU if available) and configure 4-bit quantization
# 4-bit quantization reduces model memory usage while maintaining quality
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
# Verify installed library versions for compatibility
import bitsandbytes, accelerate, transformers
bitsandbytes.__version__, accelerate.__version__, transformers.__version__

In [ ]:
# Load Gemma 2 2B instruction-tuned model for story generation
# Using 4-bit quantization to reduce memory usage
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", quantization_config=quantization_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")

In [ ]:
# Generate a short bedtime fairy tale story using the Gemma model
prompt = "generate a short, fairy tale genre, nighttime story. the response should contain only the story."

inp = tokenizer(prompt, return_tensors="pt")['input_ids'].to(model.device)
output = model.generate(inp,
                        max_new_tokens=1000)

# Decode the generated story and clean up the output
input_sentences = tokenizer.decode(output[0],
                                   skip_special_tokens=True,
                                   clean_up_tokenization_spaces=True).strip(prompt+"\n\n")

In [ ]:
input_sentences

In [ ]:
# Clean up the story: remove markdown formatting and split into paragraphs
input_sentences = input_sentences.replace("**","").split("\n\n")

In [ ]:
input_sentences

#here translation

In [ ]:
#sample story splitted into different sentences
input_sentences = [
    "When I was young, I used to go to the park every day.",
    "He has many old books, which he inherited from his ancestors.",
    "I can't figure out how to solve my problem.",
    "She is very hardworking and intelligent, which is why she got all the good marks.",
    "We watched a new movie last week, which was very inspiring.",
    "If you had met me at that time, we would have gone out to eat.",
    "She went to the market with her sister to buy a new sari.",
    "Raj told me that he is going to his grandmother's house next month.",
    "All the kids were having fun at the party and were eating lots of sweets.",
    "My friend has invited me to his birthday party, and I will give him a gift.",
]

In [ ]:
# Configure 4-bit quantization for translation model
# Using bfloat16 for better numerical stability during translation
quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

In [ ]:
# Load IndicTrans2 model for English-to-Indian language translation
# This is a 1B parameter model optimized for translating to Indian languages
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

model_trans_name = "ai4bharat/indictrans2-en-indic-1B"

tokenizer_trans = AutoTokenizer.from_pretrained(model_trans_name, trust_remote_code=True)

# Load with 4-bit quantization and low CPU memory usage
model_trans = AutoModelForSeq2SeqLM.from_pretrained(
    model_trans_name,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=quantization_config,
).to(device)

model_trans.eval()

In [ ]:
# Translate story paragraphs from English to Malayalam
# Process in batches for efficient memory usage
src_lang, tgt_lang = "eng_Latn", "mal_Mlym"

ip = IndicProcessor(inference=True)
BATCH_SIZE = 100

translations = []
for i in range(0, len(input_sentences), BATCH_SIZE):
    batch = input_sentences[i : i + BATCH_SIZE]

    # Preprocess batch and handle entity mappings
    batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

    # Tokenize and prepare inputs for model
    inputs = tokenizer_trans(
        batch,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        return_attention_mask=True,
    ).to(device)

    # Generate translations using beam search
    with torch.no_grad():
        generated_tokens = model_trans.generate(
            **inputs,
            use_cache=True,
            min_length=0,
            max_length=256,
            num_beams=5,
            num_return_sequences=1,
        )

    # Decode generated tokens to text
    generated_tokens = tokenizer_trans.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    # Post-process translations including entity replacement
    translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

    # Free up GPU memory
    del inputs
    torch.cuda.empty_cache()

In [ ]:
# Combine all translated paragraphs into a single story
translations = ''.join(translations)
translations

In [ ]:
# Split Malayalam story into chunks suitable for text-to-speech
# Each chunk is max 100 characters to ensure natural pacing in audio
story = translations

import nltk
from nltk.tokenize import sent_tokenize

from indicnlp import common
from indicnlp import loader

nltk.download('punkt')

# Set the path to the Indic NLP resources
INDIC_RESOURCES_PATH = "/content/indic_nlp_resources"

# Initialize Indic NLP library
common.set_resources_path(INDIC_RESOURCES_PATH)
loader.load()

from indicnlp.tokenize import sentence_tokenize

def split_text(text):
    # Language code for Malayalam
    lang = 'ml'
    sentences = sentence_tokenize.sentence_split(text, lang)
    return sentences

# Split the story into sentences
sentences = split_text(story)

# Group sentences into chunks to optimize TTS processing
chunks = []
current_chunk = ""
for sentence in sentences:
    if len(current_chunk) + len(sentence) <= 100:
        current_chunk += " " + sentence
    else:
        chunks.append(current_chunk.strip()+',')
        current_chunk = sentence
if current_chunk:
    chunks.append(current_chunk.strip()+',')

print(chunks)

#here tts

In [ ]:
# Configure quantization for TTS model
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
# Load Indic Parler TTS model for converting text to natural-sounding speech
# Parler TTS allows voice description to control speaking style
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, set_seed

model_tts = ParlerTTSForConditionalGeneration.from_pretrained("ai4bharat/indic-parler-tts").to(device)
tokenizer_tts = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts")
description_tokenizer = AutoTokenizer.from_pretrained(model_tts.config.text_encoder._name_or_path)

# Text to convert to speech
prompt = chunks

# Voice description: defines speaking style (calm, soothing, slow pace, no background noise)
description = len(chunks)*["Anjali's is telling a bedtime story and her voice is calm & soothing with gentle pacing and has no background noise."]

# Tokenize voice descriptions and story chunks
description_input_ids = description_tokenizer(description, return_tensors="pt", padding=True, add_special_tokens=True).to(device)
prompt_input_ids = tokenizer_tts(prompt, return_tensors="pt", padding=True, add_special_tokens=True).to(device)

# Generate audio with fixed seed for reproducibility
set_seed(0)
generation = model_tts.generate(input_ids=description_input_ids.input_ids,
                            attention_mask=description_input_ids.attention_mask,
                            prompt_input_ids=prompt_input_ids.input_ids,
                            prompt_attention_mask=prompt_input_ids.attention_mask,
                            do_sample=True,
                            return_dict_in_generate=True)

In [ ]:
# Combine generated audio chunks into a single output file
from pathlib import Path
from pydub import AudioSegment
import soundfile as sf

combined = AudioSegment.empty()

# Process each generated audio chunk
for i in range(len(chunks)):
    # Extract audio array from generation output
    audio_arr = generation.sequences[i, :generation.audios_length[i]].cpu().numpy().squeeze()
    
    # Save to temporary WAV file
    temp_file = f"/content/temp.wav"
    sf.write(temp_file, audio_arr.astype(float), model_tts.config.sampling_rate)

    # Load and concatenate with combined audio
    audio_segment = AudioSegment.from_wav(temp_file)
    combined += audio_segment

    # Clean up temporary file
    Path(temp_file).unlink()

# Export the complete bedtime story as audio
combined.export("output.wav", format="wav")

In [ ]:
# Play the generated bedtime story audio
from IPython.display import Audio, display

# Load and display audio player
Audio("output.wav", rate=model_tts.config.sampling_rate)